# 03 - Model Evaluation
**Social Media Mood Analyzer**  
This notebook evaluates the trained Logistic Regression model on the test set.
Metrics computed: Accuracy, Precision, Recall, F1-Score (overall and per mood class).

## 1. Import Libraries

In [ ]:
import pandas as pd
import os
import joblib
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

MOOD_LABELS = ['happy', 'sad', 'angry', 'neutral']
os.makedirs('results/reports', exist_ok=True)

print('Libraries imported successfully.')

## 2. Load Processed Data and Trained Model

In [ ]:
# Load processed dataset
df = pd.read_csv('data/processed/mood_data.csv')
print(f'Dataset loaded: {len(df)} records')
print(f'Mood distribution:\n{df["mood"].value_counts()}')

In [ ]:
from sklearn.model_selection import train_test_split

X = df['cleaned_text']
y = df['mood']

# Use same random_state as training to get the same test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Test set size: {len(X_test)} samples')

In [ ]:
# Load saved model and vectorizer
model      = joblib.load('results/model.pkl')
vectorizer = joblib.load('results/vectorizer.pkl')

print('Model and vectorizer loaded.')

## 3. Generate Predictions

In [ ]:
X_test_tfidf = vectorizer.transform(X_test)
y_pred       = model.predict(X_test_tfidf)

print(f'Predictions generated for {len(y_pred)} samples.')

## 4. Evaluation Metrics

In [ ]:
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1        = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print('=' * 45)
print('         EVALUATION RESULTS')
print('=' * 45)
print(f'  Accuracy  : {accuracy  * 100:.2f}%')
print(f'  Precision : {precision * 100:.2f}%')
print(f'  Recall    : {recall    * 100:.2f}%')
print(f'  F1-Score  : {f1        * 100:.2f}%')

## 5. Per-Class Classification Report

In [ ]:
report = classification_report(
    y_test, y_pred,
    target_names=MOOD_LABELS,
    zero_division=0
)
print(report)

## 6. Save Results

In [ ]:
# Save classification report as text file
report_path = 'results/reports/classification_report.txt'
with open(report_path, 'w') as f:
    f.write('SOCIAL MEDIA MOOD ANALYZER - EVALUATION REPORT\n')
    f.write('=' * 45 + '\n\n')
    f.write(f'Accuracy  : {accuracy  * 100:.2f}%\n')
    f.write(f'Precision : {precision * 100:.2f}%\n')
    f.write(f'Recall    : {recall    * 100:.2f}%\n')
    f.write(f'F1-Score  : {f1        * 100:.2f}%\n\n')
    f.write(report)
print(f'Report saved to: {report_path}')

# Save summary as CSV
summary = pd.DataFrame([{
    'Accuracy':  round(accuracy,  4),
    'Precision': round(precision, 4),
    'Recall':    round(recall,    4),
    'F1-Score':  round(f1,        4),
}])
summary.to_csv('results/reports/evaluation_summary.csv', index=False)
print('Summary saved to: results/reports/evaluation_summary.csv')
summary